# Circular Task Analysis — Assignment Notebook
## Report

This report presents the main issues we encountered during the development and analysis process, along with the underlying causes and the solutions we implemented. For each problem, we detail what went wrong, why it happened, and how we resolved it in order to ensure a correct and reliable workflow.

<details>
<summary><strong> Summary of Issues Encountered</strong></summary>

### 1. [Time Conversion](#1-time-conversion)
- [Problem](#problem-time)
- [Cause](#cause-time)
- [Resolution](#resolution-time)

### 2. [X and Y Direction](#2-x-and-y-direction)
- [Problem](#problem-sign)
- [Cause](#cause-sign)
- [Resolution](#resolution-sign)

### 3. [Formulas in the Markers Table](#3-formulas-in-the-markers-table)
- [Problem](#problem-formulas)
- [Cause](#cause-formulas)
- [Resolution](#resolution-formulas)

### 4. [Retrieve the Values from the Markers Table](#4-retrieve-the-values-from-the-markers-table)
- [Problem](#problem-values)
- [Cause](#cause-values)
- [Resolution](#resolution-values)

### 5. [Mask with Our Data](#5-mask-with-our-data)
- [Problem](#problem-mask)
- [Cause](#cause-mask)
- [Resolution](#resolution-mask)

### 6. [The initial organization of the Git repository](#6-the-initial-organization-of-the-git-repository)
- [Problem](#problem-mask)
- [Cause](#cause-mask)
- [Resolution](#resolution-mask)

</details>


## 1. Time Conversion
### <a id="problem-time"></a>Problem


### <a id="cause-time"></a>Cause
### <a id="resolution-time"></a>Resolution

## 2. X and Y Direction
### <a id="problem-sign"></a>Problem
### <a id="cause-sign"></a>Cause
### <a id="resolution-sign"></a>Resolution

## 3. Formulas in the Markers Table
### <a id="problem-formulas"></a>Problem
The marker table only contained variable names (e.g., nLaps, Re, Te, IDe) without any formulas.
We needed to reconstruct all metrics from scratch.
### <a id="cause-formulas"></a>Cause
No formulas were provided in the HTML table.
We only had the names of the variables and their numeric output, but not the computation rules behind them.

Some metrics were easy to guess (mean radius, standard deviation, etc.),
but others especially Te were non-standard and much harder to infer.
### <a id="resolution-formulas"></a>Resolution
Two strategies were used:
	•	Using the poster + AI assistance, we reconstructed most formulas
	•	For the most complex variable (Te), I validated the formula with Tifenn, who helped confirm that the correct expression was:
    Te = sigma * sqrt(2 * pi * e)
    With this, all computed values finally aligned with the marker table.

## 4. Retrieve the Values from the Markers Table
### <a id="problem-values"></a>Problem
Our computed values (Re, Te, MT, IPe, etc.) did not match the values shown in the HTML marker table, even though the formulas seemed correct.
### <a id="cause-values"></a>Cause
We initially computed all metrics using all data points inside each record, including the beginning of the recording where the cursor was not yet inside the target.

However, the original software (MouseReMoCo) applies an internal rule:

 Statistics are computed only AFTER the first entry into the target.

We were mistakenly including the “approach” phase, which greatly distorted:
	•	Re (radius too variable)
	•	Te (variance too high)
	•	nLaps (wrong angular integration)
	•	MT/lap
	•	IPe
	•	Error%
	•	Be

### <a id="resolution-values"></a>Resolution
With Lucas (another classmate) we realized the time series had to be filtered like this: 
inside = (in_rec == 1)
first_inside_idx = np.where(inside)[0][0]

t = t[first_inside_idx:]
x = x[first_inside_idx:]
y = y[first_inside_idx:]
in_rec = in_rec[first_inside_idx:]
Once we applied this filtering, all computed values finally matched the marker table.

## 5. Mask with Our Data
### <a id="problem-mask"></a>Problem
Between two records (e.g., Record 1 end → Record 2 start), the trajectory contained noise samples, which created overlapping plots and incorrect values.
### <a id="cause-mask"></a>Cause
The raw CSV contains continuous recording data, not 5 isolated segments.
So between each “Record” and “Pause”, there are background movements and noise.

If these were not removed, they interfered with:
	•	plot aesthetics
	•	nLaps
	•	MT/lap
	•	error estimates
	•	inside/outside segmentation

### <a id="resolution-mask"></a>Resolution
We masked each record separately using the timestamps extracted from the marker file:
mask = (timestamps >= start) & (timestamps <= end)
This ensured that each record only contained the exact 20-second task window, with no undesired samples.

After masking, the trajectory plots became clean, and metric computations became fully consistent with the recorded values.

## 6. The initial organization of the Git repository
### <a id="problem-mask"></a>Problem
The branch names and notebook titles were unclear, and the overall structure differed from one branch to another. This inconsistency made the repository difficult to navigate and prevented us from maintaining a clean and well-organized project.

### <a id="cause-mask"></a>Cause
The problem was caused by unclear naming choices at the start and by a lack of communication within the group.

### <a id="resolution-mask"></a>Resolution
We renamed the branches using simpler names (Figures, Markers, Report) and agreed on a working convention: creating a 001 notebook to work with the provided CSV files, and a 002 notebook for our own data exports.